In [0]:
import os
import requests
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

In [0]:
CATALOG = "gharchive"
SCHEMA = "bronze_dev"
VOLUME = "landing_files"
LANDING_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"

In [0]:
dates = [
    "2020-06-08",
    "2020-06-09",
    "2020-06-10",
    "2020-06-11",
    "2020-06-12",
    "2020-06-13",
    "2020-06-14"
]

In [0]:
urls = [
    f"https://data.gharchive.org/{date}-{hour}.json.gz"
    for date in dates
    for hour in range(24)
]


In [0]:
print(urls)

In [0]:
def download_file(url, target_dir):
    filename = url.split("/")[-1]
    local_path = os.path.join(target_dir, filename)
    tmp_path = local_path + ".tmp"
    # Skip if exists
    if os.path.exists(local_path):
        return f"SKIPPED: {filename} already exists."
        
    try:
        response = requests.get(url, stream=True, timeout=30)
        if response.status_code == 404:
            return f"MISSING (404): {url}"
        response.raise_for_status()
        
        with open(tmp_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        os.rename(tmp_path, local_path)
        return f"SUCCESS: Downloaded {filename}"
        
    except Exception as e:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return f"FAILED: {url} due to error: {str(e)}"

In [0]:
print(f"Starting download for {dates} into {LANDING_DIR}...")

In [0]:
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(download_file, url, LANDING_DIR): url for url in urls}
    for future in as_completed(futures):
        print(future.result())

In [0]:
files = dbutils.fs.ls("/Volumes/gharchive/bronze_dev/landing_files/")
print(len(files))

In [0]:
files = dbutils.fs.ls("/Volumes/gharchive/bronze_dev/landing_files/")
missing = []

all_expected = [
    f"{date}-{hour}.json.gz"
    for date in [
        "2020-06-08","2020-06-09","2020-06-10",
        "2020-06-11","2020-06-12","2020-06-13","2020-06-14"
    ]
    for hour in range(24)
]

downloaded = [f.name for f in files]
missing = [f for f in all_expected if f not in downloaded]

print(f"Downloaded: {len(downloaded)}")
print(f"Missing: {len(missing)}")
print("Missing files:")
for f in missing:
    print(f)

In [0]:
files = dbutils.fs.ls("/Volumes/gharchive/bronze_dev/landing_files/")
print(files[:5])